# Colab GPU workflow
This notebook contains only GPU-heavy stages. Data auditing, general preprocessing, plots, and report writing stay local. All expensive cells are disabled by default.

In [ ]:
# Run after cloning the repository into /content/CS_769_Project.
%cd /content/CS_769_Project
%pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import random
import torch

from data_pipeline import load_config, load_jsonl
from models import FixedAnswerer, FrozenTextEncoder, build_actor_critic, create_embedding_cache, load_embedding_cache
from training import (build_environment_factory, export_inference_bundle, mean_greedy_evidence_f1,
                      mean_greedy_episode_reward, set_seed, train_supervised, train_ppo)

assert torch.cuda.is_available(), 'Select a GPU runtime in Colab first.'
config = load_config('config.yaml')
drive_root = Path('/content/drive/MyDrive/CS_769_Project')
config['paths']['processed_dir'] = str(drive_root / 'data' / 'processed')
config['paths']['output_dir'] = str(drive_root / 'outputs')
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))

## 1. Full embedding cache

In [ ]:
RUN_EMBEDDINGS = False
embedding_dir = Path(config['paths']['output_dir']) / 'embeddings'
if RUN_EMBEDDINGS:
    encoder = FrozenTextEncoder(
        config['models']['evidence_encoder'],
        config['models']['evidence_encoder_revision'],
        device='cuda',
    )
    counts = create_embedding_cache(encoder, config['paths']['processed_dir'], embedding_dir)
    print(counts)

## 2. Supervised selector training

In [ ]:
RUN_SUPERVISED = False
if RUN_SUPERVISED:
    evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
    train_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'train_core.jsonl')
    val_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'train_val.jsonl')
    question_ids = [row['question_id'] for row in train_rows]
    validation_count = min(config['training']['validation_examples'], len(val_rows))
    validation_rng = random.Random(config['data']['split_seed'])
    val_ids = validation_rng.sample([row['question_id'] for row in val_rows], validation_count)
    env_factory, _ = build_environment_factory(
        train_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    val_factory, _ = build_environment_factory(
        val_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    for seed in config['training']['seeds']:
        set_seed(seed)
        model = build_actor_critic(config).to(device)
        run_dir = Path(config['paths']['output_dir']) / 'colab_runs' / f'supervised_seed_{seed}'
        history = train_supervised(
            model, env_factory, question_ids, epochs=config['training']['supervised_epochs'],
            learning_rate=config['training']['supervised_learning_rate'], device=device,
            validation_fn=lambda current: mean_greedy_evidence_f1(current, val_factory, val_ids, device),
            batch_size=config['training']['supervised_batch_size'], output_dir=run_dir,
        )
        run_dir.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), run_dir / 'best_model.pt')
        (run_dir / 'history.json').write_text(json.dumps(history, indent=2))

## 3. PPO pilot and full runs
Do not enable until preprocessing, embeddings, baselines, supervised checkpoints, and local PPO sanity checks are complete.

In [ ]:
RUN_PPO = False
PPO_VARIANTS = ['answer_only', 'evidence_only', 'combined', 'combined_step']
if RUN_PPO:
    from evaluation import AnswerRewardCache
    evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
    train_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'train_core.jsonl')
    val_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'train_val.jsonl')
    examples_by_id = {row['question_id']: row for row in train_rows + val_rows}
    question_ids = [row['question_id'] for row in train_rows]
    _, train_evidence = build_environment_factory(
        train_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    _, val_evidence = build_environment_factory(
        val_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    evidence_by_id = {**train_evidence, **val_evidence}
    answerer = FixedAnswerer(
        config['models']['answer_model'], config['models']['answer_model_revision'],
        device='cuda', max_input_tokens=config['models']['max_input_tokens'],
        max_output_tokens=config['models']['max_output_tokens']
    )
    answer_reward = AnswerRewardCache(
        Path(config['paths']['output_dir']) / 'answer_reward_cache.json', answerer,
        examples_by_id, evidence_by_id, config['models']['answer_model']
    )
    for variant in PPO_VARIANTS:
        for seed in config['training']['seeds']:
            set_seed(seed)
            model = build_actor_critic(config).to(device)
            supervised_path = Path(config['paths']['output_dir']) / 'colab_runs' / f'supervised_seed_{seed}' / 'best_model.pt'
            model.load_state_dict(torch.load(supervised_path, map_location=device, weights_only=True))
            reward_callback = None if variant == 'evidence_only' else answer_reward
            env_factory, _ = build_environment_factory(
                train_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
                config, answer_reward_fn=reward_callback, reward_variant=variant
            )
            val_factory, _ = build_environment_factory(
                val_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
                config, answer_reward_fn=reward_callback, reward_variant=variant
            )
            run_dir = Path(config['paths']['output_dir']) / 'colab_runs' / f'ppo_{variant}_seed_{seed}'
            validation_count = min(config['training']['validation_examples'], len(val_rows))
            validation_rng = random.Random(config['data']['split_seed'])
            validation_ids = validation_rng.sample([row['question_id'] for row in val_rows], validation_count)
            train_ppo(
                model, env_factory, question_ids, config, run_dir, device,
                validation_fn=lambda current: mean_greedy_episode_reward(
                    current, val_factory, validation_ids, device
                ),
            )

## 4. Final answer generation and GPU timing
Run only after the checkpoint is selected using `train_val`.

In [ ]:
RUN_FINAL_EVALUATION = False
FINAL_VARIANT = 'combined_step'
FINAL_SEED = 42
if RUN_FINAL_EVALUATION:
    from evaluation import (generate_answers, prepare_direct_inputs, prepare_policy_inputs,
                            prepare_similarity_inputs, save_jsonl)

    evidence_embeddings, question_embeddings = load_embedding_cache(embedding_dir)
    dev_rows = load_jsonl(Path(config['paths']['processed_dir']) / 'questions' / 'dev.jsonl')
    dev_factory, evidence_by_id = build_environment_factory(
        dev_rows, config['paths']['processed_dir'], evidence_embeddings, question_embeddings,
        config, reward_variant='evidence_only'
    )
    answerer = FixedAnswerer(
        config['models']['answer_model'], config['models']['answer_model_revision'],
        device='cuda', max_input_tokens=config['models']['max_input_tokens'],
        max_output_tokens=config['models']['max_output_tokens']
    )
    result_dir = Path(config['paths']['output_dir']) / 'results'
    baseline_inputs = {
        'direct': prepare_direct_inputs(dev_rows, config['paths']['processed_dir']),
        'similarity': prepare_similarity_inputs(dev_rows, dev_factory, evidence_by_id),
    }
    for name, inputs in baseline_inputs.items():
        save_jsonl(result_dir / f'{name}.jsonl', generate_answers(answerer, inputs, batch_size=16))
    run_specs = [('supervised', None, seed) for seed in config['training']['seeds']]
    run_specs += [('ppo', variant, seed) for variant in PPO_VARIANTS for seed in config['training']['seeds']]
    for kind, variant, seed in run_specs:
        model = build_actor_critic(config).to(device)
        if kind == 'supervised':
            path = Path(config['paths']['output_dir']) / 'colab_runs' / f'supervised_seed_{seed}' / 'best_model.pt'
            state = torch.load(path, map_location=device, weights_only=True)
            name = f'supervised_seed_{seed}'
        else:
            path = Path(config['paths']['output_dir']) / 'colab_runs' / f'ppo_{variant}_seed_{seed}' / 'best.pt'
            state = torch.load(path, map_location=device, weights_only=False)['model']
            name = f'ppo_{variant}_seed_{seed}'
        model.load_state_dict(state)
        inputs = prepare_policy_inputs(model, dev_rows, dev_factory, evidence_by_id, device)
        save_jsonl(result_dir / f'{name}.jsonl', generate_answers(answerer, inputs, batch_size=16))
    print(result_dir)

## 5. Export final local inference bundle

In [ ]:
RUN_EXPORT = False
if RUN_EXPORT:
    import numpy as np
    final_model = build_actor_critic(config).to(device)
    checkpoint_path = Path(config['paths']['output_dir']) / 'colab_runs' / f'ppo_{FINAL_VARIANT}_seed_{FINAL_SEED}' / 'best.pt'
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    final_model.load_state_dict(checkpoint['model'])
    evidence_matrix = np.load(embedding_dir / 'evidence_embeddings.npy')
    evidence_index = json.loads((embedding_dir / 'evidence_index.json').read_text())
    preprocessing_manifest = json.loads(
        (Path(config['paths']['processed_dir']) / 'preprocessing_manifest.json').read_text()
    )
    bundle_dir = Path(config['paths']['output_dir']) / 'final_bundle'
    manifest = export_inference_bundle(
        bundle_dir, final_model, config, evidence_matrix, evidence_index, preprocessing_manifest
    )
    print(json.dumps(manifest, indent=2)[:3000])